# Problem A â€” Incident Severity Triage: Visualizations

Replicates and extends all charts shown in the Streamlit **Problem A** page.

| Section | Charts |
|---------|--------|
| Performance Dashboard | Confusion matrix, severity distribution overlay, per-class metrics |
| Feature Analysis | Top feature importances (gain), feature value distributions |
| Error Analysis | Ordinal error distribution, confidence vs accuracy |
| Calibration | Max-probability histogram by predicted class |

**Pre-req:** Run `scripts/run_phase2.py` to generate `models/severity_lgbm.txt`.

In [5]:
import os
from pathlib import Path

# Find project root
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

Working directory: e:\OSFDA


In [6]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    cohen_kappa_score, confusion_matrix, classification_report,
    mean_absolute_error
)

SEVERITY_LABELS = {0: "Minor", 1: "Moderate", 2: "Substantial", 3: "Critical"}
SEVERITY_COLORS = ["#2ecc71", "#f39c12", "#e67e22", "#e74c3c"]

MODELS = Path("models")
PROCESSED = Path("data/processed")
print("Libraries loaded.")

Libraries loaded.


## 1. Load Model & Run Test-Set Predictions

Mirrors `load_severity_test_predictions()` in `streamlit_app/utils/loaders.py`.

In [7]:
import lightgbm as lgb
import joblib

from src.data.loader import load_raw_data
from src.data.target_engineering import apply_severity_rubric
from src.data.leakage_audit import get_problem_a_features
from src.features.temporal import extract_temporal_features, create_temporal_split, get_split_data
from src.features.encoding import identify_column_types, prepare_for_lgbm, bucket_experience
from src.utils.config import load_main_config, load_feature_whitelist

config   = load_main_config()
whitelist = load_feature_whitelist()

df = load_raw_data(config)
df = apply_severity_rubric(df)

# Parse time/date
td = pd.to_numeric(df["Time_Date"], errors="coerce").astype("Int64")
df["year"]  = td // 100
df["month"] = td % 100

df = create_temporal_split(df)
df = extract_temporal_features(df)
df = bucket_experience(df)

prob_a_feats = get_problem_a_features(df, whitelist)
derived      = ["year", "month", "quarter", "month_sin", "month_cos", "time_of_day_bucket"]
if "experience_bucket" in df.columns:
    derived.append("experience_bucket")
feature_cols = list(set(prob_a_feats + [d for d in derived if d in df.columns]))
feature_cols = [c for c in feature_cols if c != "Time_Date"]

col_types = identify_column_types(df[feature_cols])
for hm in col_types["high_missing"]:
    if hm in feature_cols:
        feature_cols.remove(hm)

df = prepare_for_lgbm(df, col_types["categorical"], col_types["numeric"], col_types["medium_missing"])
splits = get_split_data(df, "severity_level", feature_cols)
X_test, y_test = splits["test"]

model       = lgb.Booster(model_file=str(MODELS / "severity_lgbm.txt"))
calibrators = joblib.load(MODELS / "severity_calibrators.joblib")

X_test.columns = [col.replace(' ', '_') for col in X_test.columns]
X_test = X_test[model.feature_name()]

raw_probs = model.predict(X_test)
cal_probs = np.column_stack([c.predict(raw_probs[:, i]) for i, c in enumerate(calibrators)])
row_sums  = cal_probs.sum(axis=1, keepdims=True)
cal_probs = cal_probs / np.where(row_sums == 0, 1, row_sums)
y_pred    = cal_probs.argmax(axis=1)
y_true    = y_test.values

print(f"Test set size : {len(y_true):,}")
print(f"Severity dist : {pd.Series(y_true).value_counts().sort_index().to_dict()}")

Loaded 38655 records from E:\OSFDA\data\raw\asrs_full.parquet
Test set size : 13,309
Severity dist : {0: 4704, 1: 3201, 2: 3784, 3: 1620}


## 2. Performance Metrics

In [8]:
qwk      = cohen_kappa_score(y_true, y_pred, weights="quadratic")
mae      = mean_absolute_error(y_true, y_pred)
report   = classification_report(y_true, y_pred, output_dict=True)
macro_f1 = report["macro avg"]["f1-score"]

summary = pd.DataFrame({
    "Metric": ["QWK (Primary)", "Ordinal MAE", "Macro-F1", "Test Samples"],
    "Value":  [f"{qwk:.4f}", f"{mae:.3f}", f"{macro_f1:.3f}", f"{len(y_true):,}"],
    "Target": ["â‰¥ 0.45", "lower = better", "higher = better", "â€”"]
})
print(summary.to_string(index=False))

# Per-class table
rows = []
for i in range(4):
    r = report.get(str(i), {})
    rows.append({
        "Level": f"{i} â€” {SEVERITY_LABELS[i]}",
        "Precision": round(r.get("precision", 0), 3),
        "Recall":    round(r.get("recall", 0), 3),
        "F1":        round(r.get("f1-score", 0), 3),
        "Support":   int(r.get("support", 0))
    })
print("\nPer-class metrics:")
print(pd.DataFrame(rows).set_index("Level").to_string())

       Metric  Value          Target
QWK (Primary) 0.2164        â‰¥ 0.45
  Ordinal MAE  0.904  lower = better
     Macro-F1  0.352 higher = better
 Test Samples 13,309             â€”

Per-class metrics:
                   Precision  Recall     F1  Support
Level                                               
0 â€” Minor            0.474   0.670  0.555     4704
1 â€” Moderate         0.279   0.152  0.197     3201
2 â€” Substantial      0.473   0.515  0.493     3784
3 â€” Critical         0.249   0.122  0.164     1620


## 3. Confusion Matrix

In [9]:
cm = confusion_matrix(y_true, y_pred)

fig = px.imshow(
    cm,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=[f"Pred {SEVERITY_LABELS[i]}" for i in range(4)],
    y=[f"True {SEVERITY_LABELS[i]}" for i in range(4)],
    color_continuous_scale="Blues",
    text_auto=True,
    title="Confusion Matrix â€” Severity Triage (Test Set 2019â€“2020)",
)
fig.update_layout(height=450, width=600)
fig.show()

## 4. Predicted vs Actual Severity Distribution

In [10]:
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=y_true, name="Actual",
    opacity=0.7, marker_color="coral",
    xbins=dict(start=-0.5, end=3.5, size=1)
))
fig.add_trace(go.Histogram(
    x=y_pred, name="Predicted",
    opacity=0.7, marker_color="steelblue",
    xbins=dict(start=-0.5, end=3.5, size=1)
))
fig.update_layout(
    barmode="overlay",
    title="Predicted vs Actual Severity Distribution (Test Set)",
    xaxis=dict(
        title="Severity Level",
        tickvals=[0, 1, 2, 3],
        ticktext=[f"{i} â€“ {SEVERITY_LABELS[i]}" for i in range(4)]
    ),
    yaxis_title="Count",
    height=380,
)
fig.show()

## 5. Top Feature Importances (Gain)

In [11]:
importance = model.feature_importance(importance_type="gain")
names      = model.feature_name()
top_n      = 20
top_idx    = np.argsort(importance)[::-1][:top_n]

fig = px.bar(
    x=importance[top_idx],
    y=[names[i].replace("_", " ") for i in top_idx],
    orientation="h",
    title=f"Top {top_n} Feature Importances (Gain) â€” Severity LightGBM",
    labels={"x": "Importance (gain)", "y": "Feature"},
    color=importance[top_idx],
    color_continuous_scale="Blues",
)
fig.update_layout(height=550, showlegend=False, margin=dict(l=280))
fig.update_yaxes(autorange="reversed")
fig.show()

## 6. Split Feature Importance: Split Count vs Gain

In [12]:
gain_imp   = model.feature_importance(importance_type="gain")
split_imp  = model.feature_importance(importance_type="split")
feat_names = model.feature_name()

imp_df = pd.DataFrame({
    "feature": feat_names,
    "gain":    gain_imp,
    "split":   split_imp,
}).query("gain > 0").nlargest(25, "gain")

fig = px.scatter(
    imp_df,
    x="split",
    y="gain",
    text="feature",
    title="Feature Importance: Gain vs. Split Count (top 25 by gain)",
    labels={"split": "Split Count", "gain": "Total Gain"},
    color="gain",
    color_continuous_scale="Blues",
)
fig.update_traces(textposition="top center", textfont_size=9)
fig.update_layout(height=500, showlegend=False)
fig.show()

## 7. Ordinal Error Distribution

How far off is the model? `error = y_pred - y_true`. Zero is perfect; Â±1 is one step off.

In [13]:
errors = y_pred - y_true
error_counts = pd.Series(errors).value_counts().sort_index()

fig = px.bar(
    x=error_counts.index,
    y=error_counts.values,
    color=error_counts.index,
    color_continuous_scale="RdBu",
    title="Ordinal Error Distribution (predicted âˆ’ actual)",
    labels={"x": "Ordinal Error (pred âˆ’ true)", "y": "Count"},
)
fig.update_layout(height=350, showlegend=False)
fig.show()

print(f"Exact matches (error=0): {(errors==0).sum():,} ({(errors==0).mean()*100:.1f}%)")
print(f"Off by 1 or less:        {(np.abs(errors)<=1).sum():,} ({(np.abs(errors)<=1).mean()*100:.1f}%)")
print(f"Large errors (|err|>1):  {(np.abs(errors)>1).sum():,} ({(np.abs(errors)>1).mean()*100:.1f}%)")

Exact matches (error=0): 5,785 (43.5%)
Off by 1 or less:        9,752 (73.3%)
Large errors (|err|>1):  3,557 (26.7%)


## 8. Model Confidence Distribution by Predicted Class

Distribution of `max(calibrated_probability)` per predicted class.

In [14]:
max_conf = cal_probs.max(axis=1)

fig = go.Figure()
for level in range(4):
    mask = y_pred == level
    if mask.sum() == 0:
        continue
    fig.add_trace(go.Violin(
        y=max_conf[mask],
        name=f"Level {level} â€” {SEVERITY_LABELS[level]}",
        box_visible=True,
        meanline_visible=True,
        line_color=SEVERITY_COLORS[level],
    ))
fig.update_layout(
    title="Model Confidence Distribution by Predicted Severity Level",
    yaxis_title="Max Calibrated Probability",
    height=420,
)
fig.show()

## 9. Accuracy vs Confidence Calibration

Reliability diagram: does the model's confidence match its true accuracy?

In [15]:
bins = np.linspace(0.25, 1.0, 16)  # confidence buckets
bin_centers, bin_accs, bin_counts = [], [], []

for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (max_conf >= lo) & (max_conf < hi)
    if mask.sum() == 0:
        continue
    acc = (y_pred[mask] == y_true[mask]).mean()
    bin_centers.append((lo + hi) / 2)
    bin_accs.append(acc)
    bin_counts.append(mask.sum())

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=bin_centers, y=bin_accs, mode="lines+markers",
    name="Model accuracy",
    marker=dict(size=[max(5, c // 50) for c in bin_counts], color="steelblue"),
    line=dict(color="steelblue", width=2),
))
fig.add_trace(go.Scatter(
    x=[0.25, 1.0], y=[0.25, 1.0], mode="lines",
    name="Perfect calibration",
    line=dict(color="gray", dash="dash"),
))
fig.update_layout(
    title="Reliability Diagram â€” Confidence vs Accuracy",
    xaxis_title="Mean Predicted Confidence",
    yaxis_title="Actual Accuracy",
    height=400,
)
fig.show()

## 10. Severity Distribution in Train / Val / Test Splits

In [16]:
sev_targets = pd.read_parquet(PROCESSED / "severity_targets.parquet")
sev_splits  = pd.read_parquet(PROCESSED / "temporal_splits.parquet")
sev_merged  = sev_splits.merge(sev_targets, on="acn_num_ACN", how="inner")

split_dist = (
    sev_merged.groupby(["split", "severity_level"])
    .size()
    .reset_index(name="count")
)
split_dist["label"] = split_dist["severity_level"].map(SEVERITY_LABELS)

fig = px.bar(
    split_dist,
    x="split",
    y="count",
    color="label",
    barmode="group",
    color_discrete_sequence=SEVERITY_COLORS,
    title="Severity Level Distribution Across Train / Val / Test Splits",
    labels={"split": "Split", "count": "Count", "label": "Severity"},
    category_orders={"split": ["train", "val", "test"]},
)
fig.update_layout(height=400)
fig.show()

# Year trend
year_dist = sev_merged.groupby(["year", "severity_level"]).size().reset_index(name="count")
year_dist["label"] = year_dist["severity_level"].map(SEVERITY_LABELS)
fig2 = px.line(
    year_dist, x="year", y="count", color="label",
    color_discrete_sequence=SEVERITY_COLORS,
    markers=True,
    title="Severity Distribution Over Time (all splits)",
    labels={"year": "Year", "count": "Reports", "label": "Severity"},
)
fig2.update_layout(height=380)
fig2.show()